# Full-variable autoregressive anomaly-detection baseline

This mirrors TEP notebook 3.0. Each endogenous process variable is predicted from the lagged history of all six variables; only the three endogenous-variable residuals contribute to the anomaly score. Healthy training runs fit the models and residual precision matrix; faulty validation runs select the moving-average window, consecutive alarms, and threshold. The selected detector is then evaluated on every faulty and healthy held-out test run.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_baseline_comparison

baseline_results = run_baseline_comparison("full_var", PROJECT_ROOT)
print("Full-variable baseline fit, validation-tuned, and evaluated on held-out test runs.")

Full-variable baseline fit, validation-tuned, and evaluated on held-out test runs.


## 2. Validation and hyperparameter selection

As in TEP, thresholds are the maximum smoothed score observed in the healthy first phase of faulty validation runs. Window and consecutive-alarm settings maximize trajectory-phase F1, with median detection delay breaking ties.

In [2]:
display(baseline_results["validation_grid"].sort_values(
    ["Trajectory-Phase F1", "Median Anomaly Detection Delay"],
    ascending=[False, True],
).reset_index(drop=True))
display(baseline_results["selected_settings"])

,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,8,5,2.983853,0.513493,0.399772,0.0,0.571196,0.904444,0.808889,0.0,0.894349,182,225,31.0
1,8,6,2.983853,0.508123,0.393761,0.0,0.565034,0.897778,0.795556,0.0,0.886139,179,225,30.0
2,7,5,3.231822,0.501511,0.384989,0.0,0.555945,0.895556,0.791111,0.0,0.883375,178,225,30.5
3,8,7,2.983853,0.503052,0.388133,0.0,0.559216,0.895556,0.791111,0.0,0.883375,178,225,32.0
4,6,5,3.429900,0.491735,0.372928,0.0,0.543259,0.895556,0.791111,0.0,0.883375,178,225,32.5
5,7,6,3.231822,0.496047,0.378878,0.0,0.549545,0.893333,0.786667,0.0,0.880597,177,225,33.0
6,8,8,2.983853,0.498175,0.382756,0.0,0.553613,0.893333,0.786667,0.0,0.880597,177,225,34.0
7,7,7,3.231822,0.490847,0.373106,0.0,0.543448,0.888889,0.777778,0.0,0.875000,175,225,34.0
8,6,6,3.429900,0.486126,0.366650,0.0,0.536568,0.884444,0.768889,0.0,0.869347,173,225,33.0
9,7,8,3.231822,0.485831,0.367572,0.0,0.537554,0.862222,0.724444,0.0,0.840206,163,225,33.0


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components
0,full_var,2,8,5,2.983853,NaN


## 3. Test on faulty time series

The validation-selected settings are fixed for the full faulty test set.

In [3]:
display(baseline_results["faulty_test_metrics"])
display(baseline_results["faulty_test_by_type"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,full_var,2,8,5,2.983853,0.53101,0.421383,0.0,0.59292,0.917778,0.835556,0.0,0.910412,188,225,31.5


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,41,0.546667,194.0
1,reaction_coupling_shift,75,75,1.000000,19.0
2,thermal_coupling_shift,75,72,0.960000,36.0


## 4. Test on fault-free time series

The same fixed threshold is evaluated on all healthy held-out runs. The comparison table is updated as each of notebooks 3.0, 3.1, and 3.2 is run.

In [4]:
display(baseline_results["healthy_test_metrics"])
display(baseline_results["comparison"])

,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,full_var,2,8,5,2.983853,1.0,0.0,0.0,0.0,0.0,0,75


,Baseline,Max Lag,Window Size,Consecutive Alarms,Threshold,PCA Components,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
2,full_var,2,8,5,2.983853,NaN,0.592920,0.910412,0.835556,31.5,0.0,0.0
0,pca,2,8,5,1.882515,2.0,0.399929,0.888889,0.800000,56.0,0.0,0.0
1,self_history,2,7,5,3.407215,NaN,0.483112,0.728814,0.573333,32.0,0.0,0.0
